# CRO — datos necesarios y análisis

Los archivos originales no contienen el historial de etapas, las evaluaciones de prospectos ni las interacciones necesarios para responder las preguntas del CRO.

**Para cubrir esa información faltante, creamos datos sintéticos.** Con ellos construimos las métricas y demostramos el análisis; no comprobamos qué ocurrió realmente en la empresa.

**Recorrido:** tablas a crear → datos a generar → validación → preguntas, métricas y gráficas.


In [1]:
import csv, hashlib, json, sqlite3, sys
from contextlib import closing
from pathlib import Path
from IPython.display import HTML, Markdown, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/cro_demo.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from scripts.cro_demo import FIELDS, DICTIONARY, CSV_NAMES, model_html, preview_table
from scripts.cro_complete import validate_full
ESCENARIO = 'referencia'
assert ESCENARIO in ('referencia', 'conversion', 'demora')
BASE = ROOT / 'resultados/cro'
CARPETA = ROOT / 'datos/sinteticos/cro' / ESCENARIO
DB_CARPETA = ROOT / 'datos/bases/cro' / ESCENARIO
METRICAS = BASE / 'metricas' / ESCENARIO
manifest = json.loads((BASE / 'manifest.json').read_text())
display(Markdown(f'**Escenario:** {ESCENARIO}. Puedes cambiarlo arriba; empezamos con referencia.'))


**Escenario:** referencia. Puedes cambiarlo arriba; empezamos con referencia.

## 1. Tablas y datos que creamos

### Tablas creadas

- **Cuentas (empresas potenciales):** empresas que entran y sus características.
- **Registros de etapas:** entradas, salidas y resultados en cada etapa.
- **Interacciones:** intentos de contacto y sus resultados.
- **Pagos:** eventos de pago y su validez.

**Etapas** es el catálogo auxiliar de nombres y orden del funnel. **Ganados** se calcula desde los registros de etapas; no es otra tabla de origen.

### Datos incluidos

- **Cuentas:** identificador, fecha y tipo de entrada, canal, campaña, industria, producto, región, perfil y evaluación de calidad.
- **Registros de etapas:** cuenta, etapa, fechas de entrada y salida, destino, estado, motivo de pérdida y asignación de responsable.
- **Interacciones:** cuenta y visita de etapa, fecha, tipo de contacto, resultado y responsable.
- **Pagos:** identificador del pago, cuenta, fecha y estado; incluir historia previa para identificar nuevos pagadores.

Las tablas se conectan por `id_cuenta`; las interacciones también usan `id_registro` para identificar la visita de etapa. En esta demo, una cuenta representa una empresa con una oportunidad comercial.

A continuación revisamos qué representa cada fila y una muestra de los datos creados.


In [2]:
display(HTML('<style>.scroll{overflow-x:auto}.scroll table{border-collapse:collapse}.scroll td,.scroll th{padding:8px;border-bottom:1px solid #ddd;text-align:left;white-space:nowrap}details{margin:12px 0}summary{cursor:pointer}</style>'))
def leer_tabla(nombre):
    with (CARPETA / f'{CSV_NAMES.get(nombre,nombre)}.csv').open(encoding='utf-8', newline='') as f:
        reader=csv.DictReader(f)
        assert reader.fieldnames == FIELDS[nombre]
        return list(reader)
def mostrar_tabla(nombre,columnas):
    filas=leer_tabla(nombre)
    display(Markdown(f'**{len(filas):,} filas.** Muestra de 3 filas con las columnas principales.'))
    display(HTML(preview_table(columnas,[{k:v or None for k,v in r.items()} for r in filas[:3]])))
    definiciones=[dict(columna=k,significado=v[1]) for k,v in DICTIONARY[nombre][2].items()]
    detalle=preview_table(['columna','significado'],definiciones)
    archivo=CARPETA / f'{CSV_NAMES.get(nombre,nombre)}.csv'
    display(HTML('<details><summary>Ver significado de todas las columnas</summary>'+detalle+'</details>'))
    display(Markdown(f'Archivo completo: `{archivo}`'))
    return filas


### A. Cuentas: quiénes entraron

Una fila representa una **empresa potencial**. Por ejemplo: llegó por Marketing, pertenece a Comercio y entró por New. Nos permite contar entradas y comparar grupos similares.

También guarda la evaluación de calidad: si encaja en el perfil objetivo, tiene una necesidad cubierta e intención de compra. **Sin evaluación no significa mala calidad.** Los criterios ilustrativos están en el detalle de columnas y en las reglas del modelo. Para el análisis de calidad usamos una puntuación de 0 a 100, guardada por separado y vinculada a cada cuenta.


In [3]:
cuentas=mostrar_tabla('dim_cuentas',['id_cuenta','fecha_captacion','canal','industria','tipo_entrada','calificado'])


**1,800 filas.** Muestra de 3 filas con las columnas principales.

id_cuenta,fecha_captacion,canal,industria,tipo_entrada,calificado
C01-00000,2026-01-04,Marketing pago,Servicios,New,0
C01-00001,2026-01-11,Orgánico,Comercio,New,—
C01-00002,2026-01-19,Referidos,Comercio,New,0


columna,significado
id_cuenta,Identificador de la cuenta
canal,Origen de captación
industria,Sector de actividad
tipo_entrada,"Entrada por New, SQL o compra desde producto"
fecha_captacion,Primera captación observada en la simulación
campana,Campaña de captación; no aplica a todos los canales
producto,Producto de interés
region,Región ilustrativa
perfil,Tamaño de empresa ilustrativo
responsable,Responsable inicial


Archivo completo: `./datos/sinteticos/cro/referencia/dim_cuentas.csv`

### B. Registros de etapas: qué pasó en cada etapa

Una fila representa **una visita de una empresa a una etapa**. Si entra a Working el día 10 y sale el 15, estuvo cinco días en esa visita. Fechas y resultado permiten calcular conversión, pérdidas y tiempos por etapa.

`avance` significa que pasó al destino indicado; puede saltar una etapa. `perdido` terminó sin avanzar. `abierto` sigue pendiente al corte. `ganado` llegó a Won. Una empresa puede tener varias filas y volver a una etapa: **filas no equivale a clientes**.


In [4]:
registros=mostrar_tabla('fact_etapas',['id_registro','id_cuenta','id_etapa','entrada','salida','estado','id_etapa_destino'])


**6,740 filas.** Muestra de 3 filas con las columnas principales.

id_registro,id_cuenta,id_etapa,entrada,salida,estado,id_etapa_destino
C01-00000-0,C01-00000,New,2026-01-04,2026-01-10,avance,Working
C01-00000-1,C01-00000,Working,2026-01-10,2026-01-13,avance,Engaged
C01-00000-2,C01-00000,Engaged,2026-01-13,2026-01-21,avance,SQL


columna,significado
id_registro,Identificador de esta entrada
id_cuenta,Cuenta que entró
id_etapa,Etapa a la que entró
entrada,Día de entrada a la etapa
salida,Día de avance o pérdida; vacío si abierto o Won
estado,"avance, perdido, abierto o ganado"
id_etapa_destino,Siguiente etapa alcanzada; solo en avances
motivo_perdida,"Motivo registrado de pérdida, no causa demostrada"
responsable_etapa,Responsable de la visita
fecha_asignacion_etapa,Asignación durante la visita


Archivo completo: `./datos/sinteticos/cro/referencia/fact_etapas.csv`

### C. Interacciones: qué atención recibió

Una fila representa **un intento de contacto**, como una llamada o un correo. `id_registro` indica en qué visita de etapa ocurrió.

Sirve para medir cuánto tardamos en intentar atender y cuántas visitas quedaron sin intento. `sin_respuesta` es un intento; `intercambio_efectivo` implica respuesta o conversación. **Esto mide atención; las fechas de etapas miden permanencia.**


In [5]:
interacciones=mostrar_tabla('interacciones',['id_cuenta','id_registro','fecha','tipo','resultado'])


**8,635 filas.** Muestra de 3 filas con las columnas principales.

id_cuenta,id_registro,fecha,tipo,resultado
C01-00000,C01-00000-0,2026-01-09,correo,sin_respuesta
C01-00000,C01-00000-1,2026-01-10,llamada,intercambio_efectivo
C01-00000,C01-00000-1,2026-01-11,llamada,intercambio_efectivo


columna,significado
id_interaccion,Identificador de interacción
id_cuenta,Cuenta atendida
id_registro,Visita de etapa atendida
fecha,Fecha del intento
tipo,Llamada o correo
resultado,Sin respuesta o intercambio efectivo
responsable,Responsable de la interacción


Archivo completo: `./datos/sinteticos/cro/referencia/interacciones.csv`

### D. Pagos: quién pagó y cuándo

Una fila representa **un evento de pago**. Solo `valido` cuenta. Una misma empresa puede tener pagos fallidos y varios pagos válidos.

Sirve para contar nuevos pagadores y compras desde producto. Un segundo pago no crea otro cliente nuevo. La demo incluye historia previa para distinguir quienes ya pagaban. **Won es un cierre comercial; el pago se comprueba aquí.**


In [6]:
pagos=mostrar_tabla('pagos',['id_pago','id_cuenta','fecha_pago','estado_pago'])


**2,436 filas.** Muestra de 3 filas con las columnas principales.

id_pago,id_cuenta,fecha_pago,estado_pago
C01-00000-fallido,C01-00000,2026-02-06,fallido
C01-00000-p1,C01-00000,2026-02-06,valido
C01-00000-p2,C01-00000,2026-03-08,valido


columna,significado
id_pago,Identificador de pago
id_cuenta,Cuenta pagadora
fecha_pago,Fecha del evento de pago
estado_pago,Valido o fallido; solo valido cuenta


Archivo completo: `./datos/sinteticos/cro/referencia/pagos.csv`

### Catálogo auxiliar: cuáles son las etapas

Una fila por nombre de etapa. `siguiente` indica la transición habitual que vamos a medir; Won no tiene siguiente.

**Cómo se conectan:** una cuenta tiene varias visitas y pagos; cada visita puede tener varias interacciones. No unimos todo directamente porque multiplicaría filas y distorsionaría las métricas.


In [7]:
etapas=leer_tabla('dim_etapas')
display(HTML(preview_table(FIELDS['dim_etapas'],[{k:v or None for k,v in r.items()} for r in etapas])))


id_etapa,orden,siguiente
New,1,Working
Working,2,Engaged
Engaged,3,SQL
SQL,4,Demo
Demo,5,Proposal
Proposal,6,Won
Won,7,—


## 2. Validación de los datos

Control técnico: comprueba integridad de archivos, relaciones y fechas. El detalle queda plegado.


In [8]:
for relative,digest in manifest['hashes'].items():
    assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()==digest,relative
with closing(sqlite3.connect(DB_CARPETA/'modelo.sqlite')) as con:
    controles=validate_full(con,manifest['config']['cutoff'])
    for tabla in ('dim_cuentas','fact_etapas','interacciones','pagos','dim_etapas'):
        assert con.execute(f'SELECT COUNT(*) FROM {tabla}').fetchone()[0]==len(leer_tabla(tabla))
display(HTML('<details><summary>Ver resultado de la validación</summary><p>Archivos íntegros; claves, relaciones, fechas e historia de pagos coherentes.</p></details>'))


## 3. Pregunta principal y recorrido del análisis

**¿La tasa de conversión de New a Won aumentó o disminuyó después del aumento de New?**

**Preguntas priorizadas:** ¿Aumentó New? → ¿Bajó la conversión? → ¿En qué canales? → ¿Cambió la calidad? → ¿En qué etapas cayó? → ¿Qué explica la caída? Después proponemos acciones.

**Población principal:** cuentas que entraron por New. Las entradas directas a SQL se evalúan desde SQL y por separado; entrar allí no demuestra por sí solo mayor calidad. Producto queda fuera de New → Won, pero se conserva como contexto del crecimiento total.




### 3.1. ¿Cuánto creció New, frente a qué periodo y en qué segmentos?

**Por qué lo preguntamos:** comprobar con datos si New realmente aumentó, antes de buscar explicaciones.

**Qué medimos:** cuánto cambian las entradas a New frente a un periodo de referencia y en qué segmentos ocurre.

**Periodo y población:** New de enero a agosto de 2026, global y por canal; comparación con el mes anterior. Región y otros segmentos quedan pendientes si aportan al diagnóstico.

**Variación mensual:** `(New del mes − New del mes anterior) / New del mes anterior × 100`.

**Datos:** cuentas que entraron por New, fecha de entrada y atributos del segmento.


In [9]:
from scripts.cro_charts import grafica_new, CONFIG
figura_new=grafica_new(METRICAS/'demanda.csv', ESCENARIO)
figura_new.write_html(BASE/'graficas/01_new_tendencia.html',include_plotlyjs=True,config=CONFIG)
figura_new.show(renderer='plotly_mimetype',config=CONFIG)


In [10]:
def leer_metrica(nombre,dimension='total',**filtros):
    with (METRICAS/f'{nombre}.csv').open(encoding='utf-8',newline='') as f:
        return [r for r in csv.DictReader(f) if r['dimension']==dimension and all(r[k]==str(v) for k,v in filtros.items())]
demanda=leer_metrica('demanda')
columnas=['Mes','Entradas a New','Variación mensual (%)']
filas=[{'Mes':r['mes'],'Entradas a New':int(r['entradas_new']),
        'Variación mensual (%)':round(float(r['crecimiento_new_pct']),1) if r['crecimiento_new_pct'] else None}
       for r in demanda]
display(HTML('<details><summary>Ver valores de la gráfica</summary>'+preview_table(columnas,filas)+'</details>'))


Mes,Entradas a New,Variación mensual (%)
2026-01,127,—
2026-02,124,-2.4
2026-03,132,6.5
2026-04,133,0.8
2026-05,179,34.6
2026-06,183,2.2
2026-07,182,-0.5
2026-08,173,-4.9


#### Conclusión


In [11]:
# Genera la conclusión a partir de las métricas actuales.
from html import escape
from scripts.cro_insights import conclusion_new
lectura_new=conclusion_new(METRICAS/'demanda.csv')
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;">'+escape(lectura_new['texto'])+'</div>'))


#### Desglose por canal
**Por qué segmentamos:** identificar si el crecimiento de New se concentró en algún canal o fue compartido por todos.


In [12]:
from scripts.cro_charts import grafica_new_por_canal
figura_canales=grafica_new_por_canal(METRICAS/'demanda.csv')
figura_canales.show(renderer='plotly_mimetype',config=CONFIG)


#### Conclusión por canal


In [13]:
from scripts.cro_insights import conclusion_new_por_canal
lectura_canales=conclusion_new_por_canal(METRICAS/'demanda.csv')
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;">'+escape(lectura_canales['texto']).replace('\n\n','<br><br>')+'</div>'))


### 3.2. ¿Cómo entendemos la relación entre New y Won?

Medimos cuánto tardan los New en llegar a Won para entender cuándo podrían reflejarse las nuevas entradas en los cierres y establecer un plazo común para comparar la conversión.


#### ¿Cuánto tiempo tardaron los New en llegar a Won?

**Qué medimos:** la mediana de días desde New hasta Won.

**Periodo y población:** 432 cuentas que entraron por New entre enero y agosto de 2026 y llegaron a Won hasta el 30 de septiembre. Para establecer la referencia previa al aumento, analizamos por separado enero–abril.

**Métrica:** `mediana(fecha de Won − primera entrada a New)`.

**Datos:** fecha de entrada a New y fecha de Won de cada cuenta.

**Criterio:** usamos la mediana porque los casos muy largos afectan menos el resultado. Incluye únicamente cuentas ganadas; no mide cuánto tardarán las pendientes.


In [14]:
import pandas as pd
# Mediana desde las fechas de primera entrada a New y primer Won.
ganados_base=pd.read_csv(ROOT/'datos/sinteticos/cro/referencia/ganados.csv')
ganados_new=ganados_base.loc[ganados_base['tipo_entrada'].eq('New')].copy()
duracion=(pd.to_datetime(ganados_new['fecha_won'])-pd.to_datetime(ganados_new['fecha_entrada_inicial'])).dt.total_seconds()/86400
mediana_new_won=float(duracion.median())
mediana_texto=f'{mediana_new_won:g}'.replace('.',',')
assert duracion.notna().all() and duracion.ge(0).all()
ganados_new['mes_new']=pd.to_datetime(ganados_new['fecha_entrada_inicial']).dt.to_period('M').astype(str)
ganados_new['duracion_dias']=duracion
# Histórica enero-abril (previa a mayo): periodo, cuentas y resultado.
janabr=ganados_new.loc[ganados_new['mes_new'].between('2026-01','2026-04')].copy()
mediana_eneabr=float(janabr['duracion_dias'].median())
n_eneabr=int(len(janabr))
# Cierre con capacidad: general 32 incluye demora May+; histórica enero-abril 30 previa.
assert mediana_new_won == 32, 'Revisar la justificación del plazo: cambió la mediana general.'
assert mediana_eneabr == 30, 'Revisar la mediana histórica enero-abril.'
resumen_duracion=ganados_new.groupby('mes_new').agg(cuentas_ganadas=('id_cuenta','size'),mediana_dias=('duracion_dias','median')).reset_index()
resumen_duracion.columns=['Mes de entrada a New','Cuentas que llegaron a Won','Mediana (días)']
display(resumen_duracion)

,Mes de entrada a New,Cuentas que llegaron a Won,Mediana (días)
0,2026-01,38,29.5
1,2026-02,42,31.0
2,2026-03,50,29.0
3,2026-04,48,29.5
4,2026-05,61,35.0
5,2026-06,65,34.0
6,2026-07,58,33.5
7,2026-08,70,32.0


In [15]:
import plotly.graph_objects as go
figura_duracion=go.Figure(go.Scatter(x=resumen_duracion.iloc[:,0], y=resumen_duracion.iloc[:,2], mode="lines+markers", line=dict(color="#2563eb",width=3), hovertemplate="Mes de entrada: %{x}<br>Mediana: %{y} días<extra></extra>"))
figura_duracion.update_layout(title="Tiempo de New a Won por mes de entrada · 2026",template="plotly_white",height=400,font=dict(family="Arial, sans-serif",size=14),margin=dict(l=65,r=30,t=65,b=60),xaxis_title="Mes de entrada a New",yaxis_title="Mediana (días)")
figura_duracion.add_hline(y=mediana_new_won,line_dash="dot",line_color="#64748b",annotation_text=f"Mediana general: {mediana_texto} días",annotation_position="bottom right")
figura_duracion.show(renderer="plotly_mimetype",config=CONFIG)

display(HTML(f'<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;"><strong>Resultado:</strong> la mediana de New a Won es {mediana_texto} días en general (enero–agosto, {len(ganados_new)} cuentas) y 30 días entre enero y abril ({n_eneabr} cuentas). Mantenemos ese plazo histórico de 30 para comparar abril y mayo: distingue la mediana previa de 30 de la general actual de 32 (que incluye la demora May+) y conserva 30/60/90 y meses maduros.</div>'))


#### ¿Qué porcentaje de New llegó a Won en 30 días?

**Qué medimos:** qué porcentaje de los New de cada mes llegó a Won en 30 días.

**Referencia:** usamos los 30 días de la mediana histórica calculada en el punto anterior.

**Periodo y población:** New de enero a agosto de 2026, global y por canal.

**Métrica:** `New del mes que llegó a Won en 30 días / entradas a New del mes × 100`.

**Datos:** primera fecha de entrada a New y fecha de Won de cada cuenta.


In [16]:
from scripts.cro_charts import grafica_new_won
figura_new_won=grafica_new_won(METRICAS/'cohortes.csv',ESCENARIO,ventana=30)
figura_new_won.show(renderer='plotly_mimetype',config=CONFIG)

from scripts.cro_insights import conclusion_conversion
lectura_conversion=conclusion_conversion(METRICAS/'cohortes.csv',ventana=30,mes_aumento=lectura_new['evidencia']['mes'])
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;"><strong>Conclusión:</strong> '+escape(lectura_conversion['texto'].split('Marketing pago:')[0].strip())+'</div>'))


#### Desglose por canal


In [17]:
from scripts.cro_charts import grafica_conversion_por_canal
figura_conversion_canales=grafica_conversion_por_canal(METRICAS/'cohortes.csv',ventana=30)
figura_conversion_canales.show(renderer='plotly_mimetype',config=CONFIG)


#### Análisis de la conversión


In [18]:
from scripts.cro_insights import conclusion_conversion
lectura_conversion=conclusion_conversion(METRICAS/'cohortes.csv',ventana=30,mes_aumento=lectura_new['evidencia']['mes'])
resto = lectura_conversion['texto']
if 'Marketing pago:' in resto:
    resto = 'Marketing pago:'+resto.split('Marketing pago:',1)[1]
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;">'+escape(resto).replace('\n\n','<br><br>')+'</div>'))


### 3.3. ¿Cambió la calidad de los leads o el criterio para calificarlos?

**Por qué lo preguntamos:** antes de analizar las etapas, revisamos si cambió la calidad de los leads como posible explicación de la caída de conversión. La puntuación permite seguir su evolución.

**Comprobación previa:** ¿existe una puntuación de calidad de los leads?

- **Si existe:** analizar su evolución con una escala comparable.
- **Si no existe:** avanzar con los datos disponibles y proponer implementar la medición.

**Para este ejercicio:** generamos puntuaciones aleatorias de 0 a 100. No representan una evaluación real ni explican la caída de conversión.

**Qué medimos:** si cambió la puntuación media de los leads que entraron por New.

**Periodo y población:** leads que entraron por New entre enero y agosto de 2026, global y por canal; abril como referencia y mayo en adelante como foco. Todos tienen puntuación en este ejercicio.

**Métrica:** `suma de puntuaciones / número de New con puntuación`.

**Datos:** identificador de la cuenta, fecha de entrada a New, canal y puntuación de calidad.


In [19]:
from scripts.cro_quality import analyze_scores
analisis_puntuacion=analyze_scores(CARPETA/"dim_cuentas.csv",CARPETA/"puntuaciones_leads.csv",month_of_increase=lectura_new["evidencia"]["mes"])
for figura in analisis_puntuacion["figures"]:
    figura.show(renderer="plotly_mimetype",config=CONFIG)
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;"><strong>Conclusión:</strong> '+escape(analisis_puntuacion["reading"])+'</div>'))


In [20]:
from scripts.cro_cohorts import audit
base_coherente, tasas_coherentes, control_coherencia = audit(METRICAS)
assert control_coherencia['cohortes_conciliadas'] > 0


### 3.4. ¿Cambió la conversión y, si cayó, en qué transición?

**Por qué lo preguntamos:** ahora analizamos las etapas para identificar dónde disminuyó la conversión de New a Won.

**Qué medimos:** qué porcentaje avanzó de cada nivel al siguiente.

**Periodo y población:** los mismos New, con abril como referencia y mayo–agosto de 2026 como foco. Global y por canal en las dos mayores caídas.

**Métrica:** `cuentas que alcanzaron o superaron el siguiente nivel / cuentas que alcanzaron o superaron el nivel actual × 100`.

**Plazo:** 30 días desde la entrada a New, igual para todas las etapas.

**Datos:** cuentas y fechas de sus cambios de etapa.


In [21]:
from scripts.cro_stage_focus import analyze_stage_focus
foco_etapas=analyze_stage_focus(METRICAS,lectura_new["evidencia"]["mes"])
foco_etapas["global_figure"].show(renderer="plotly_mimetype",config=CONFIG)


#### Conclusión global


In [22]:
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;">'+escape(foco_etapas["global_reading"]).replace("\n", "<br>")+'</div>'))

# Control visible y plegado: mismo grupo y mismo plazo que New → Won.
control=tasa_control=foco_etapas['metrics'].loc[lambda x:x.dimension.eq('total')&x.mes.isin(['2026-04','2026-05']),['mes','etapa','elegibles','avances_siguiente','conversion_pct']].copy()
control['etapa']=pd.Categorical(control['etapa'],categories=['New','Working','Engaged','SQL','Demo','Proposal'],ordered=True)
control=control.sort_values(['mes','etapa'])
control.columns=['Mes de New','Nivel','Cuentas en el nivel','Cuentas en el siguiente nivel','Conversión (%)']
display(HTML('<details><summary>Ver cuentas y tasas conciliadas con 3.2</summary>'+control.to_html(index=False,float_format=lambda x:f'{x:.1f}')+'</details>'))


Mes de New,Nivel,Cuentas en el nivel,Cuentas en el siguiente nivel,Conversión (%)
2026-04,New,133,127,95.5
2026-04,Working,127,103,81.1
2026-04,Engaged,103,90,87.4
2026-04,SQL,90,63,70.0
2026-04,Demo,63,48,76.2
2026-04,Proposal,48,28,58.3
2026-05,New,179,167,93.3
2026-05,Working,167,138,82.6
2026-05,Engaged,138,122,88.4
2026-05,SQL,122,102,83.6


#### Profundizamos en las transiciones prioritarias

**Por qué segmentamos:** identificar si la caída se concentra en algún canal y acotar la investigación.


In [23]:
for detalle in foco_etapas["focused"]:
    detalle["figure"].show(renderer="plotly_mimetype",config=CONFIG)
    display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;"><strong>Conclusión:</strong> '+escape(detalle["reading"])+'</div>'))


### 3.5. ¿Cambió la rapidez o cobertura de atención y seguimiento?

**Por qué lo preguntamos:** investigamos qué podría explicar las caídas en Demo y Proposal. En este ejercicio evaluamos la carga por persona y la demora de atención.

**Otras opciones:** contacto efectivo (si logramos conversar con el cliente) y motivos de pérdida (por qué se perdió la oportunidad). No se desarrollan en este análisis.

#### Carga por persona disponible

**Qué medimos:** cuántas cuentas llegan a cada etapa por persona disponible.

**Periodo y población:** cuentas que llegaron a Demo y Proposal entre abril y agosto de 2026, agrupadas por mes de llegada a cada etapa; sin desglose por canal.

**Métrica:** `cuentas únicas que ingresan a la etapa en el mes / personal disponible en esa etapa y mes`.

**Datos:** registros de etapas para llegadas y `dim_capacidad` para personal. Reingresos cuentan una vez por cuenta y etapa; los saltos no generan visitas ficticias.


In [24]:
from scripts.cro_cohorts import metricas_carga
from scripts.cro_charts import grafica_carga_por_persona, CONFIG
carga_total = metricas_carga(METRICAS)
carga = carga_total.loc[carga_total['mes'].between('2026-04', '2026-08')].copy()
figura_carga = grafica_carga_por_persona(carga)
figura_carga.write_html(BASE/'graficas/02_carga_por_persona.html', include_plotlyjs=True, config=CONFIG)
figura_carga.show(renderer='plotly_mimetype', config=CONFIG)

# Conclusión y valores calculados desde las métricas (sin cifras a mano).
def fila_carga(etapa, mes):
    r = carga.loc[carga['id_etapa'].eq(etapa)&carga['mes'].eq(mes)].iloc[0]
    return int(r['llegadas']), int(r['personal_disponible']), float(r['carga_por_persona'])
demo_abr_lleg, demo_abr_pers, demo_abr_carga = fila_carga('Demo', '2026-04')
demo_may_lleg, demo_may_pers, demo_may_carga = fila_carga('Demo', '2026-05')
prop_abr_lleg, prop_abr_pers, prop_abr_carga = fila_carga('Proposal', '2026-04')
prop_may_lleg, prop_may_pers, prop_may_carga = fila_carga('Proposal', '2026-05')
fmt = lambda v: f'{v:.1f}'.replace('.', ',')
texto_carga = (
    f'De abril a mayo, la carga por persona aumentó un {fmt((demo_may_carga/demo_abr_carga-1)*100)}% en Demo '
    f'y un {fmt((prop_may_carga/prop_abr_carga-1)*100)}% en Proposal. '
    'Después se mantuvo por encima de abril, con el mismo personal. Revisamos si aumentó la demora de atención.'
)
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;"><strong>Conclusión:</strong> '+escape(texto_carga)+'</div>'))
display(HTML('<details><summary>Ver valores de la gráfica</summary>'+preview_table(
    ['Mes', 'Etapa', 'Llegaron', 'Personal', 'Carga por persona'],
    [{'Mes': r['mes'], 'Etapa': r['id_etapa'], 'Llegaron': int(r['llegadas']),
      'Personal': int(r['personal_disponible']), 'Carga por persona': round(float(r['carga_por_persona']), 1)}
     for _, r in carga.sort_values(['mes', 'id_etapa']).iterrows()])+'</details>'))


Mes,Etapa,Llegaron,Personal,Carga por persona
2026-04,Demo,79,10,7.9
2026-04,Proposal,73,8,9.1
2026-05,Demo,123,10,12.3
2026-05,Proposal,77,8,9.6
2026-06,Demo,123,10,12.3
2026-06,Proposal,106,8,13.2
2026-07,Demo,117,10,11.7
2026-07,Proposal,104,8,13.0
2026-08,Demo,113,10,11.3
2026-08,Proposal,93,8,11.6


#### Demora hasta el primer intento

**Por qué lo preguntamos:** la carga por persona aumentó. Ahora comprobamos si también aumentó el tiempo hasta el primer intento de contacto en Demo y Proposal.

**Qué medimos:** la mediana de días desde la entrada a la etapa hasta la primera llamada o correo de ventas.

**Periodo y población:** New de abril a agosto de 2026 que visitaron Demo y Proposal dentro de los 30 días desde New, sin desglose por canal. La mediana incluye solo cuentas con intento; también mostramos cuántas recibieron atención.

**Métrica:** `mediana(primer intento − entrada a la etapa)` y `cuentas con intento / cuentas con visita`.

**Datos:** registros de etapas e interacciones unidos por registro, mismas cuentas New y mismo reloj desde New que en 3.2–3.4.


In [25]:
import pandas as pd
from scripts.cro_charts import grafica_demora_focos
detalle_atencion = pd.read_csv(METRICAS/'atencion_focos_new.csv')
figura_demora = grafica_demora_focos(detalle_atencion)
figura_demora.write_html(BASE/'graficas/03_demora_primer_intento.html', include_plotlyjs=True, config=CONFIG)
figura_demora.show(renderer='plotly_mimetype', config=CONFIG)

# Conclusión y valores calculados desde las fechas (no del parámetro del generador).
def fila_demora(etapa, mes):
    r = detalle_atencion.loc[detalle_atencion['etapa'].eq(etapa)&detalle_atencion['mes'].eq(mes)&detalle_atencion['dimension'].eq('total')].iloc[0]
    return float(r['mediana_primer_intento']), int(r['cuentas_con_visita']), int(r['cuentas_con_intento'])
demo_abr_med, demo_abr_vis, demo_abr_int = fila_demora('Demo', '2026-04')
demo_may_med, demo_may_vis, demo_may_int = fila_demora('Demo', '2026-05')
prop_abr_med, prop_abr_vis, prop_abr_int = fila_demora('Proposal', '2026-04')
prop_may_med, prop_may_vis, prop_may_int = fila_demora('Proposal', '2026-05')
fmt = lambda v: f'{v:.1f}'.replace('.', ',')
texto_demora = (
    f'De abril a mayo, la mediana hasta el primer intento de contacto aumentó de {fmt(demo_abr_med)} a {fmt(demo_may_med)} días en Demo '
    f'y de {fmt(prop_abr_med)} a {fmt(prop_may_med)} días en Proposal.'
)
# Describir la evolución posterior únicamente cuando los datos la sostienen.
posterior = detalle_atencion.loc[detalle_atencion['dimension'].eq('total') & detalle_atencion['mes'].gt('2026-05')].pivot(index='mes', columns='etapa', values='mediana_primer_intento').sort_index()
if not posterior.empty:
    primer_mes = posterior.iloc[0]
    if primer_mes['Demo'] == demo_may_med and primer_mes['Proposal'] == prop_may_med:
        from scripts.cro_insights import MONTHS
        mes_estable = MONTHS[pd.Period(posterior.index[0], freq='M').month-1]
        texto_demora += f' En {mes_estable}, ambas mantuvieron esos tiempos.'
    if posterior.iloc[-1]['Demo'] < demo_may_med and posterior.iloc[-1]['Proposal'] < prop_may_med:
        texto_demora += ' Después disminuyeron en ambas etapas.'
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;"><strong>Conclusión:</strong> '+escape(texto_demora)+'</div>'))
display(HTML('<details><summary>Ver valores de la gráfica</summary>'+preview_table(
    ['Mes de New', 'Etapa', 'Con visita', 'Con intento', 'Mediana (días)'],
    [{'Mes de New': r['mes'], 'Etapa': r['etapa'], 'Con visita': int(r['cuentas_con_visita']),
      'Con intento': int(r['cuentas_con_intento']),
      'Mediana (días)': round(float(r['mediana_primer_intento']), 1) if pd.notna(r['mediana_primer_intento']) else None}
     for _, r in detalle_atencion.loc[detalle_atencion['dimension'].eq('total')].sort_values(['mes', 'etapa']).iterrows()])+'</details>'))


Mes de New,Etapa,Con visita,Con intento,Mediana (días)
2026-04,Demo,58,41,3.0
2026-04,Proposal,48,29,2.0
2026-05,Demo,96,68,4.0
2026-05,Proposal,63,29,4.0
2026-06,Demo,91,73,4.0
2026-06,Proposal,69,20,4.0
2026-07,Demo,85,64,3.0
2026-07,Proposal,59,29,4.0
2026-08,Demo,97,70,2.0
2026-08,Proposal,67,38,3.0


### 3.6. Síntesis: más volumen, misma capacidad, atención más lenta y menor conversión dentro del plazo


In [26]:
# Síntesis calculada desde métricas del escenario (sin cifras a mano).
from scripts.cro_cohorts import stage_metrics
from scripts.cro_cohorts import load_base as cargar_base
base_sintesis = cargar_base(METRICAS)
tasas_sintesis = stage_metrics(base_sintesis['cohorts'], base_sintesis['order'])
def tasa(etapa, mes):
    r = tasas_sintesis.loc[tasas_sintesis['etapa'].eq(etapa)&tasas_sintesis['mes'].eq(mes)&tasas_sintesis['ventana'].eq(30)&tasas_sintesis['dimension'].eq('total')].iloc[0]
    return float(r['conversion_pct'])
demo_conv_abr, demo_conv_may = tasa('Demo', '2026-04'), tasa('Demo', '2026-05')
prop_conv_abr, prop_conv_may = tasa('Proposal', '2026-04'), tasa('Proposal', '2026-05')
coh = pd.read_csv(METRICAS/'cohortes.csv')
coh = coh.loc[coh['ventana'].eq(30)&coh['dimension'].eq('total')]
new_abr = coh.loc[coh['mes_entrada'].eq('2026-04')].iloc[0]
new_may = coh.loc[coh['mes_entrada'].eq('2026-05')].iloc[0]
fmt1 = lambda v: f'{v:.1f}'.replace('.', ',')
# La síntesis cambia si el escenario deja de respaldar esta secuencia.
misma_capacidad = demo_abr_pers == demo_may_pers and prop_abr_pers == prop_may_pers
mayor_volumen = new_may['entradas_new'] > new_abr['entradas_new']
atencion_mas_lenta = demo_may_med > demo_abr_med and prop_may_med > prop_abr_med
menor_conversion = (demo_conv_may < demo_conv_abr and prop_conv_may < prop_conv_abr
                    and new_may['conversion_new_won_pct'] < new_abr['conversion_new_won_pct'])
if misma_capacidad and mayor_volumen and atencion_mas_lenta and menor_conversion:
    texto_sintesis = (
        'El volumen aumentó sin ampliar el personal. La atención se hizo más lenta en Demo y Proposal '
        'y cayó la conversión de New a Won dentro de 30 días. '
        'Esto muestra menor avance en ese plazo, no pérdidas definitivas de clientes.'
    )
else:
    texto_sintesis = (
        'Los indicadores no respaldan toda la secuencia de mayor volumen, mismo personal, '
        'atención más lenta y menor conversión. Revisamos los hallazgos antes de cerrar el diagnóstico. '
        'La conversión a 30 días no mide pérdidas definitivas de clientes.'
    )
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;background:#ffffff;color:#000000;"><strong>Conclusión final:</strong> '+escape(texto_sintesis)+'</div>'))


## 4. Acción y evaluación

Reforzar capacidad en Demo y Proposal (responsable propuesto: Sales). Propuesta de evaluación con periodo y criterios verificables; sin resultados inventados ni número óptimo de contrataciones.


In [27]:
# Acción propuesta y plan de medición (criterios desde la referencia de abril, sin impacto inventado).
periodo_prueba = 'octubre–noviembre de 2026 (corte 2026-12-31, 30 días desde New)'
criterios = (
    f'mediana primer intento Demo ≤ {fmt1(demo_abr_med)} días y Proposal ≤ {fmt1(prop_abr_med)} días; '
    f'carga Demo < {fmt1(demo_may_carga)} y Proposal < {fmt1(prop_may_carga)} con volumen similar a mayo; '
    f'conversión Demo→Proposal ≥ {fmt1(demo_conv_abr)}%, Proposal→Won ≥ {fmt1(prop_conv_abr)}% y '
    f'New→Won ≥ {fmt1(float(new_abr["conversion_new_won_pct"]))}% (referencia abril).'
)
filas_accion = [
    {'Hallazgo': f'Demo: carga {fmt1(demo_abr_carga)}→{fmt1(demo_may_carga)} y demora {fmt1(demo_abr_med)}→{fmt1(demo_may_med)} días; conversión {fmt1(demo_conv_abr)}%→{fmt1(demo_conv_may)}%.',
     'Acción propuesta': 'Reforzar capacidad en Demo.', 'Responsable': 'Sales (propuesto)',
     'Evaluación': f'{periodo_prueba}: ¿bajan demora y carga y sube conversión? {criterios}'},
    {'Hallazgo': f'Proposal: carga {fmt1(prop_abr_carga)}→{fmt1(prop_may_carga)} y demora {fmt1(prop_abr_med)}→{fmt1(prop_may_med)} días; conversión {fmt1(prop_conv_abr)}%→{fmt1(prop_conv_may)}%.',
     'Acción propuesta': 'Reforzar capacidad en Proposal.', 'Responsable': 'Sales (propuesto)',
     'Evaluación': f'{periodo_prueba}: ¿bajan demora y carga y sube conversión? {criterios}'},
]
# Tabla breve; los criterios comunes se muestran una sola vez debajo.
filas_presentacion = [
    {'Etapa': 'Demo', 'Acción': 'Reforzar capacidad', 'Responsable': 'Sales',
     'Evaluación': f'Demora ≤ {fmt1(demo_abr_med)} días; carga < {fmt1(demo_may_carga)} cuentas/persona; conversión a Proposal ≥ {fmt1(demo_conv_abr)}%.'},
    {'Etapa': 'Proposal', 'Acción': 'Reforzar capacidad', 'Responsable': 'Sales',
     'Evaluación': f'Demora ≤ {fmt1(prop_abr_med)} días; carga < {fmt1(prop_may_carga)} cuentas/persona; conversión a Won ≥ {fmt1(prop_conv_abr)}%.'},
]
columnas_accion = ['Etapa', 'Acción', 'Responsable', 'Evaluación']
def fila_md(fila):
    return '| ' + ' | '.join(str(fila[k]).replace('|', '—').replace('\n', ' ') for k in columnas_accion) + ' |'
tabla_md = '| ' + ' | '.join(columnas_accion) + ' |\n|---|---|---|---|\n' + '\n'.join(fila_md(f) for f in filas_presentacion)
display(Markdown(tabla_md))
display(Markdown(
    f'**Periodo de evaluación:** {periodo_prueba}.\n\n'
    f'**Resultado global esperado:** conversión New → Won ≥ {fmt1(float(new_abr["conversion_new_won_pct"]))}%, con volumen similar a mayo. '
    'Los objetivos por etapa toman abril como referencia; la carga se compara con mayo.'
))


| Etapa | Acción | Responsable | Evaluación |
|---|---|---|---|
| Demo | Reforzar capacidad | Sales | Demora ≤ 3,0 días; carga < 12,3 cuentas/persona; conversión a Proposal ≥ 76,2%. |
| Proposal | Reforzar capacidad | Sales | Demora ≤ 2,0 días; carga < 9,6 cuentas/persona; conversión a Won ≥ 58,3%. |

**Periodo de evaluación:** octubre–noviembre de 2026 (corte 2026-12-31, 30 días desde New).

**Resultado global esperado:** conversión New → Won ≥ 21,1%, con volumen similar a mayo. Los objetivos por etapa toman abril como referencia; la carga se compara con mayo.